# 📧 Spam Email Detector — v2
A deep learning classifier that detects spam SMS messages using a Bidirectional LSTM neural network.

**Improvements over v1:**
- Fixed `Input(shape=())` mismatch that could cause runtime errors
- Added `ReduceLROnPlateau` for smarter learning rate decay
- Added precision-recall threshold sweep to find the optimal decision boundary
- Added normalized confusion matrix view
- Added vocabulary inspection cell
- Google Drive mount before training (checkpoints survive runtime resets)
- `share=True` on Gradio launch for a real public URL
- Fallback so Step 8 works even if Step 7 crashes

**Run all cells in order:** Runtime → Run all (`Ctrl+F9`)

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow pandas scikit-learn seaborn gradio --quiet

import pandas as pd
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (TextVectorization, Embedding,
                                      Bidirectional, LSTM,
                                      GlobalMaxPool1D, Dense, Dropout)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

print('✅ All libraries imported successfully!')
print(f'   TensorFlow version: {tf.__version__}')

## Step 1b — Mount Google Drive (keeps checkpoints safe)

In [ ]:
# This saves your model checkpoints to Drive so they survive a runtime reset.
# If you are running locally, set SAVE_DIR to any local folder and skip the mount.
import os

USE_DRIVE = True   # ← set False if running locally

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/spam_detector'
else:
    SAVE_DIR = '.'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Checkpoints will be saved to: {SAVE_DIR}')

## Step 2 — Load the Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df  = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print('📊 Dataset Overview')
print('=' * 40)
print(df.head())
print(f'\nTotal messages : {len(df)}')
print(f'Spam           : {(df["label"] == "spam").sum()}  ({(df["label"] == "spam").mean():.1%})')
print(f'Ham            : {(df["label"] == "ham").sum()}  ({(df["label"] == "ham").mean():.1%})')

## Step 3 — Preprocess the Text

In [ ]:
# --- Configuration Constants ---
MAX_WORDS  = 10000
MAX_LENGTH = 150

# Encode labels: spam=1, ham=0
df['label_enc'] = (df['label'] == 'spam').astype(int)

# Train / test split using raw text strings
X_train, X_test, y_train, y_test = train_test_split(
    df['message'].values,
    df['label_enc'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label_enc']
)

# Class weights to handle imbalance (87% ham vs 13% spam)
weights       = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}

print('✅ Split and class balancing complete!')
print(f'   Train samples : {len(X_train)}')
print(f'   Test samples  : {len(X_test)}')
print(f'   Class weights : Ham={class_weights[0]:.2f}, Spam={class_weights[1]:.2f}')

## Step 4 — Build the Model (Bidirectional LSTM)

In [ ]:
# --- TextVectorization layer ---
vectorizer_layer = TextVectorization(
    max_tokens=MAX_WORDS,
    output_sequence_length=MAX_LENGTH,
    output_mode='int'
)
vectorizer_layer.adapt(X_train)

# --- Model architecture ---
model = Sequential([
    # FIX: shape=() accepts a scalar string per sample (not shape=(1,)).
    # shape=(1,) reshapes inputs to (batch, 1), which causes a rank mismatch
    # inside TextVectorization that expects a 1D string tensor per sample.
    tf.keras.Input(shape=(), dtype=tf.string, name='raw_text_input'),

    # Vectorization: text → integer token sequence
    vectorizer_layer,

    # Embedding: integers → 128-d meaning vectors (mask_zero ignores padding)
    Embedding(input_dim=MAX_WORDS, output_dim=128, mask_zero=True, name='masked_embedding'),

    # Bidirectional LSTM: reads the sentence forward AND backward
    Bidirectional(LSTM(64, return_sequences=True), name='bidirectional_lstm'),

    # Global Max Pooling: grabs the strongest spam signal from the sequence
    GlobalMaxPool1D(name='global_max_pooling'),

    # Regularization + Dense layers
    Dropout(0.4, name='dropout_1'),
    Dense(32, activation='relu', name='dense_feature_extractor'),
    Dropout(0.2, name='dropout_2'),

    # Output: a single probability between 0 (ham) and 1 (spam)
    Dense(1, activation='sigmoid', name='output_prediction_node')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

model.summary()

## Step 5 — Train the Model

In [ ]:
# --- Callbacks ---

# Stops training when val_loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Saves the best model to Drive (survives runtime resets)
checkpoint = ModelCheckpoint(
    filepath=f'{SAVE_DIR}/spam_detector_best.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# NEW: Halves the LR when val_loss plateaus for 2 epochs.
# This lets Adam overshoot early (fast learning) then refine
# carefully later — often squeezes an extra 0.5–1% accuracy.
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# --- Training loop ---
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    class_weight=class_weights,
    callbacks=[early_stop, checkpoint, lr_scheduler],
    verbose=1
)

print(f'\n✅ Training ended at epoch {len(history.history["loss"])}')

## Step 6 — Evaluate the Model

In [ ]:
# --- 1. Standard metrics ---
metrics_eval = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy  : {metrics_eval[1]:.4f}')
print(f'Test Precision : {metrics_eval[2]:.4f}')
print(f'Test Recall    : {metrics_eval[3]:.4f}\n')

y_prob = model.predict(X_test, verbose=0).ravel()   # raw probabilities (keep for threshold sweep)
y_pred = (y_prob > 0.5).astype(int)                 # default threshold

print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# --- 2. Training history plots ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# NEW: learning rate trace — shows when ReduceLROnPlateau fired
axes[2].plot(history.history.get('lr', [1e-3] * len(history.history['loss'])), color='tab:orange')
axes[2].set_title('Learning rate')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

# --- 3. Confusion matrices: counts AND normalised side-by-side ---
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'], ax=axes[0])
axes[0].set_title('Confusion matrix (counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# NEW: normalised view — each row sums to 100%, making class-level error rates obvious
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'], ax=axes[1])
axes[1].set_title('Confusion matrix (row %)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Step 6b — Precision-Recall Threshold Sweep

The default `> 0.5` threshold is rarely optimal. For spam detection we usually
care most about **not blocking legitimate messages** (high precision), so we want
the threshold that maximises F1 — or a higher threshold if false positives are
especially costly in your deployment.

In [ ]:
precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob)

# F1 at each threshold (precision_recall_curve returns one extra point; trim it)
f1_vals = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (
    precision_vals[:-1] + recall_vals[:-1] + 1e-8
)

best_idx       = np.argmax(f1_vals)
best_threshold = thresholds[best_idx]
best_f1        = f1_vals[best_idx]

print(f'Optimal threshold : {best_threshold:.3f}')
print(f'Best F1 score     : {best_f1:.4f}')
print(f'Precision at best : {precision_vals[best_idx]:.4f}')
print(f'Recall at best    : {recall_vals[best_idx]:.4f}\n')

# Precision-recall curve plot with the best threshold marked
plt.figure(figsize=(8, 4))
plt.plot(recall_vals[:-1], precision_vals[:-1], lw=2, label='PR curve')
plt.scatter(recall_vals[best_idx], precision_vals[best_idx],
            color='red', zorder=5, s=80,
            label=f'Best threshold ({best_threshold:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Re-evaluate using the optimal threshold
y_pred_opt = (y_prob > best_threshold).astype(int)
print('Classification report with optimal threshold:')
print(classification_report(y_test, y_pred_opt, target_names=['Ham', 'Spam']))

## Step 6c — Vocabulary Inspection

In [ ]:
# The vectorizer sorts tokens by frequency (most common first).
# Indices 0 and 1 are reserved for padding and unknown tokens.
# This shows the tokens the model actually learned from.
vocab = vectorizer_layer.get_vocabulary()

print(f'Vocabulary size   : {len(vocab)}')
print(f'\nTop 30 tokens (most frequent in training data):')
print(vocab[2:32])   # skip 0='', 1='[UNK]'

print(f'\nRare tokens (tail of vocabulary, index 9990-9999):')
print(vocab[9990:])

## Step 7 — Save the Model

In [ ]:
save_path = f'{SAVE_DIR}/spam_detector.keras'
model.save(save_path)
print(f'✅ Model saved to {save_path}')

reloaded_model = load_model(save_path)
print('✅ Reloaded model is ready!')

## Step 8 — Test with Your Own Messages

In [ ]:
# FIX: fall back to the in-memory model if Step 7 failed (e.g. disk full)
inference_model = reloaded_model if 'reloaded_model' in dir() else model

# Use the threshold found in Step 6b instead of the hardcoded 0.5
THRESHOLD = best_threshold if 'best_threshold' in dir() else 0.5
print(f'Using decision threshold: {THRESHOLD:.3f}\n')

def predict_spam(text, threshold=THRESHOLD):
    prob  = inference_model.predict([text], verbose=0)[0][0]
    label = 'SPAM' if prob > threshold else 'HAM'
    print(f'Message : {text[:70]}')
    print(f'Result  : {label}  (confidence: {prob:.2%})\n')

predict_spam("Congratulations! You've won a FREE iPhone. Click here now!")
predict_spam("Hey, are we still meeting for lunch tomorrow?")
predict_spam("URGENT: Your account will be suspended. Verify NOW: bit.ly/x")
predict_spam("Can you send me the lecture slides from today?")
predict_spam("You have been selected for a cash prize of $1000. Call us now!")

## Step 9 — Interactive Web App (Gradio)

In [ ]:
import gradio as gr

THRESHOLD = best_threshold if 'best_threshold' in dir() else 0.5

def check_spam(text):
    if not text.strip():
        return 'Please enter a message.'
    prob  = inference_model.predict([text], verbose=0)[0][0]
    label = 'SPAM' if prob > THRESHOLD else 'HAM'
    return f'{label} — confidence: {prob:.2%}  (threshold: {THRESHOLD:.2f})'

# FIX: share=True generates a real public Gradio URL (valid for 72 h)
# so you can test from a phone or share the link with others.
# Remove share=True if you only need a local URL.
gr.Interface(
    fn=check_spam,
    inputs=gr.Textbox(lines=3, placeholder='Type a message to check...', label='Message'),
    outputs=gr.Textbox(label='Result'),
    title='📧 Spam Detector v2',
    description=(
        f'Bidirectional LSTM spam classifier. '
        f'Decision threshold auto-tuned to {THRESHOLD:.2f} via precision-recall sweep.'
    ),
    examples=[
        ['Congratulations! You won a FREE prize. Call now!'],
        ['Are you coming to class tomorrow?'],
        ['URGENT: Your bank account is locked. Verify identity at bit.ly/fakebank']
    ]
).launch(share=True)
